# Chapter 3: Summary Statistics

Companion notebook for *Master Exploratory Data Analysis: A Hands-On Beginner's Guide with Python, Statistics & A/B Testing*.

Run the cells from top to bottom (**Kernel → Restart Kernel and Run All Cells**). The data files are in the same folder as this notebook.

## 3.1 One Summary for a Whole Column

In [1]:
import numpy as np
import pandas as pd

customers = pd.read_csv('customers.csv', parse_dates=['signup_date'])
orders = pd.read_csv('orders.csv', parse_dates=['order_date'])
plans = pd.read_csv('plans.csv')
cancellations = pd.read_csv('cancellations.csv')
survey = pd.read_csv('survey.csv')
add_ons = orders[orders['order_type'] == 'Add-on']
plans

,plan,monthly_price,bags_per_month
0,Basic,16.0,1
1,Standard,28.0,2
2,Premium,49.0,4


### `df.describe()`

In [2]:
customers['age'].describe().round(2)

count    4043.00
mean       37.96
std        10.81
min        18.00
25%        30.00
50%        38.00
75%        45.00
max        76.00
Name: age, dtype: float64

In [3]:
add_ons['amount'].describe().round(2)

count    4520.00
mean       25.82
std        34.68
min         4.60
25%        13.69
50%        19.26
75%        30.21
max       635.74
Name: amount, dtype: float64

In [4]:
customers.describe().round(1)

,customer_id,signup_date,age
count,4210.0,4210,4043.0
mean,3105.5,2025-02-16 06:53:11.258907,38.0
min,1001.0,2024-01-01 00:00:00,18.0
25%,2053.2,2024-09-07 00:00:00,30.0
50%,3105.5,2025-02-28 00:00:00,38.0
75%,4157.8,2025-08-23 00:00:00,45.0
max,5210.0,2025-12-31 00:00:00,76.0
std,1215.5,NaN,10.8


### Choosing which percentiles to show

In [5]:
orders['delivery_days'].describe(percentiles=[0.1, 0.5, 0.9]).round(2)

count    28772.00
mean         3.30
std          1.58
min          1.00
10%          1.00
50%          3.00
90%          5.00
max         13.00
Name: delivery_days, dtype: float64

## 3.2 The Middle: Mean, Median and Mode

### `s.mean()`

In [6]:
orders['delivery_days'].mean()

np.float64(3.2978937856249133)

In [7]:
mean_days = orders['delivery_days'].mean()
print(f'Average delivery time: {mean_days:.2f} days')

Average delivery time: 3.30 days


### `s.median()`

In [8]:
median_days = orders['delivery_days'].median()
print(f'Median delivery time: {median_days:.0f} days')

Median delivery time: 3 days


In [9]:
amount_mean = add_ons['amount'].mean()
amount_median = add_ons['amount'].median()
print(f'Mean add-on amount:   ${amount_mean:.2f}')
print(f'Median add-on amount: ${amount_median:.2f}')

Mean add-on amount:   $25.82
Median add-on amount: $19.26


In [10]:
gifts = add_ons['items'] >= 10
print(f'Gift orders: {gifts.sum()}')

regular = add_ons[~gifts]
print(f"Mean without gifts:   ${regular['amount'].mean():.2f}")
print(f"Median without gifts: ${regular['amount'].median():.2f}")

Gift orders: 24
Mean without gifts:   $23.64
Median without gifts: $19.19


### `s.mode()`

In [11]:
print(customers['plan'].mode())

0    Basic
Name: plan, dtype: str


In [12]:
common_plan = customers['plan'].mode().iloc[0]
common_days = orders['delivery_days'].mode().iloc[0]
common_age = customers['age'].mode().iloc[0]
print(f'Most common plan:          {common_plan}')
print(f'Most common delivery time: {common_days} days')
print(f'Most common age:           {common_age:.0f}')

Most common plan:          Basic
Most common delivery time: 3 days
Most common age:           18


### Ratings: the mean, the median or the mode?

In [13]:
satisfaction = survey['satisfaction']
print(satisfaction.value_counts().sort_index())
print(f'Mean:   {satisfaction.mean():.2f}')
print(f'Median: {satisfaction.median():.0f}')
print(f'Mode:   {satisfaction.mode().iloc[0]}')

satisfaction
1      5
2     29
3    194
4    232
5     72
Name: count, dtype: int64
Mean:   3.63
Median: 4
Mode:   4


## 3.3 Spread: How Far Values Wander

### `s.std()`

In [14]:
age = customers['age']
print(f'Mean age: {age.mean():.1f}')
print(f'Std of age: {age.std():.1f}')

Mean age: 38.0
Std of age: 10.8


In [15]:
low = age.mean() - 2 * age.std()
high = age.mean() + 2 * age.std()
within = age.between(low, high).sum() / age.count()
print(f'From {low:.1f} to {high:.1f} years: {within:.1%} of customers')

From 16.3 to 59.6 years: 97.2% of customers


In [16]:
amount = add_ons['amount']
print(f'Mean: ${amount.mean():.2f}, std: ${amount.std():.2f}')

Mean: $25.82, std: $34.68


### `s.var()`

In [17]:
print(f'Variance of age: {age.var():.1f}')
print(f'Std of age:      {age.std():.1f}')
print(f'Std squared:     {age.std() ** 2:.1f}')

Variance of age: 116.9
Std of age:      10.8
Std squared:     116.9


### Range and `s.quantile()`

In [18]:
print(f"Delivery days: {orders['delivery_days'].min()} to "
      f"{orders['delivery_days'].max()}")

Delivery days: 1 to 13


In [19]:
days = orders['delivery_days']
days.quantile([0.5, 0.9, 0.95, 0.99])

0.50    3.0
0.90    5.0
0.95    6.0
0.99    8.0
Name: delivery_days, dtype: float64

In [20]:
q1 = amount.quantile(0.25)
q3 = amount.quantile(0.75)
print(f'Add-on amount: middle half from ${q1:.2f} to ${q3:.2f}')
print(f'IQR: ${q3 - q1:.2f}')

Add-on amount: middle half from $13.69 to $30.21
IQR: $16.52


### `np.percentile()`

In [21]:
ages = customers['age'].dropna()
np.percentile(ages, [10, 50, 90])

array([24., 38., 52.])

## 3.4 Shape in Two Numbers

### `s.skew()`

In [22]:
print(f'Skew of age: {age.skew():.2f}')

Skew of age: 0.19


### `s.kurt()`

In [23]:
print(f'Kurtosis of age: {age.kurt():.2f}')

Kurtosis of age: -0.37


In [24]:
shape = pd.concat({
    'age': age.agg(['skew', 'kurt']),
    'delivery days': days.agg(['skew', 'kurt']),
    'add-on amount': amount.agg(['skew', 'kurt']),
}, axis=1)
shape.round(2)

,age,delivery days,add-on amount
skew,0.19,0.71,11.96
kurt,-0.37,0.51,175.87


## 3.5 Summarizing Text Columns

### `describe(include=)`

In [25]:
customers[['plan', 'channel', 'region']].describe()

,plan,channel,region
count,4210,4210,4210
unique,3,5,4
top,Basic,Search,West
freq,1887,1472,1281


In [26]:
customers[['plan', 'age']].describe(include='all').round(1)

,plan,age
count,4210,4043.0
unique,3,NaN
top,Basic,NaN
freq,1887,NaN
mean,NaN,38.0
std,NaN,10.8
min,NaN,18.0
25%,NaN,30.0
50%,NaN,38.0
75%,NaN,45.0


In [27]:
customers.describe(include='str')

,city,state,region,channel,plan,device
count,4210,4210,4210,4210,4210,4210
unique,20,17,4,5,3,2
top,New York,TX,West,Search,Basic,Mobile
freq,543,544,1281,1472,1887,2597


In [28]:
cancellations['reason'].describe()

count              2369
unique                5
top       Too expensive
freq                647
Name: reason, dtype: object

## 3.6 Finding the Biggest and the Smallest

### `s.idxmax()` and `s.idxmin()`

In [29]:
biggest = add_ons['amount'].idxmax()
add_ons.loc[biggest]

order_id                      109145
customer_id                     2010
order_date       2025-01-20 00:00:00
order_type                    Add-on
items                             25
amount                        635.74
discount                         0.0
delivery_days                      2
Name: 9144, dtype: object

In [30]:
smallest = add_ons['amount'].idxmin()
add_ons.loc[smallest, ['order_date', 'items', 'amount']]

order_date    2024-12-13 00:00:00
items                           1
amount                        4.6
Name: 7317, dtype: object

In [31]:
orders_region = orders.merge(customers[['customer_id', 'region']],
                             on='customer_id', validate='many_to_one')
by_region = orders_region.groupby('region')['delivery_days'].mean()
by_region.round(2)

region
Midwest      3.07
Northeast    2.79
South        3.31
West         3.88
Name: delivery_days, dtype: float64

In [32]:
slowest = by_region.idxmax()
fastest = by_region.idxmin()
print(f'Slowest: {slowest} ({by_region.max():.2f} days)')
print(f'Fastest: {fastest} ({by_region.min():.2f} days)')

Slowest: West (3.88 days)
Fastest: Northeast (2.79 days)


## 3.7 Rates and Weighted Averages

### A rate is the mean of True and False

In [33]:
customers['cancelled'] = customers['customer_id'].isin(
    cancellations['customer_id'])
customers[['customer_id', 'plan', 'cancelled']].head()

,customer_id,plan,cancelled
0,1001,Basic,False
1,1002,Basic,True
2,1003,Standard,True
3,1004,Basic,True
4,1005,Basic,True


In [34]:
n_cancelled = customers['cancelled'].sum()
cancel_rate = customers['cancelled'].mean()
print(f'Cancelled: {n_cancelled} of {len(customers)}')
print(f'Cancellation rate: {cancel_rate:.1%}')

Cancelled: 2369 of 4210
Cancellation rate: 56.3%


In [35]:
by_plan = customers.groupby('plan')['cancelled'].agg(
    cancel_rate='mean', customers='count')
by_plan.round(3)

,cancel_rate,customers
plan,,
Basic,0.699,1887
Premium,0.373,812
Standard,0.494,1511


### `np.average()`

In [36]:
simple = by_plan['cancel_rate'].mean()
print(f'Simple average of the three plan rates: {simple:.1%}')

Simple average of the three plan rates: 52.2%


In [37]:
weighted = np.average(by_plan['cancel_rate'],
                      weights=by_plan['customers'])
print(f'Weighted by customers: {weighted:.1%}')

Weighted by customers: 56.3%


In [38]:
region_orders = orders_region.groupby('region')['delivery_days'].count()
simple_days = by_region.mean()
weighted_days = np.average(by_region, weights=region_orders)
print(f'Plain average of the 4 regional averages: {simple_days:.3f}')
print(f'Weighted by orders per region:           {weighted_days:.3f}')
print(f'Average of every order:                  {mean_days:.3f}')

Plain average of the 4 regional averages: 3.264
Weighted by orders per region:           3.298
Average of every order:                  3.298


In [39]:
customers_per_plan = customers.groupby('plan')['customer_id'].count()
plans['customers'] = plans['plan'].map(customers_per_plan)
plans

,plan,monthly_price,bags_per_month,customers
0,Basic,16.0,1,1887
1,Standard,28.0,2,1511
2,Premium,49.0,4,812


In [40]:
simple_price = plans['monthly_price'].mean()
weighted_price = np.average(plans['monthly_price'],
                            weights=plans['customers'])
print(f'Plain average of the plan prices: ${simple_price:.2f}')
print(f'Average price per customer:       ${weighted_price:.2f}')

Plain average of the plan prices: $31.00
Average price per customer:       $26.67
